# 01 · Data and validation

**Question:** What can these splits establish about unseen community rules?

The expanded study develops on four policies and keeps two entire policy types reserved. The original two-policy experiments remain historical evidence. A protected boundary improves the design; it does not itself establish generalization.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Historical references: 2,029 rows, two policies; expanded research is a separate cohort.")
print("Aggregate checksums verified. This notebook performs no model fitting.")
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records, heldout=False):
    return pd.DataFrame([{"Representation": r["model"], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]}
        for r in records if not heldout or r["protocol"] == "heldout_rule"]).round(4)


In [ ]:
import sys
sys.path.insert(0, str(root / "scripts"))
from build_research_report import display_figure
from build_release_report import display_boundary
from jigsaw_rules.features import feature_evidence
from jigsaw_rules.research import research_evidence
from jigsaw_rules.diagnostics import diagnostic_evidence
from jigsaw_rules.pairs import pairs_evidence
from jigsaw_rules.robustness import robustness_evidence
from jigsaw_rules.gate import feature_gate
from jigsaw_rules.instructions import instruction_evidence
from jigsaw_rules.released import released_evidence
from jigsaw_rules.expanded import expanded_evidence
from jigsaw_rules.retrieval import retrieval_evidence
from jigsaw_rules.resolution import resolution_evidence
from jigsaw_rules.formatting import formatting_evidence
from jigsaw_rules.feature_decision import decision_evidence
from build_expanded_report import display_figure as display_expanded
from build_formatting_report import display_figure as display_formatting

expanded = expanded_evidence(root)
retrieval = retrieval_evidence(root)
resolution = resolution_evidence(root)
formatting = formatting_evidence(root)
decision = decision_evidence(root)
assert expanded is not None and retrieval is not None and resolution is not None
assert formatting is not None and decision is not None
controls = feature_evidence(root)
research = research_evidence(root)
sensitivity = diagnostic_evidence(root)
pairs = pairs_evidence(root)
robustness = robustness_evidence(root)
assert all(item is not None for item in (controls, research, sensitivity, pairs, robustness))
gate = feature_gate(root)
instructions = instruction_evidence(root)
released = released_evidence(root)
assert released is not None
print("Verified expanded study:", expanded["metadata"]["run_id"])


## A public source, with a prospective research reserve
The host released six-policy evaluation data after the competition. Source version, archive/member hashes and original train/preview parity are verified. The partition uses policy, Public/Private metadata and normalized text before research targets are materialized. Historical exposure removes 54 reserved rows; research body/support overlap removes 1,323 research rows. No model-selection function opens the released solution file.

In [ ]:
display_boundary(root)
print("Protected rows:", released["boundary"]["role_counts"]["confirmation"])
print("Target access during feature research:", expanded["audit"]["confirmation_labels_accessed"])

## Policy prevalence and repeated annotations
Class balance differs sharply by policy, so pooled accuracy or pooled AUC can mislead. Equal-weight policy AUC keeps the policy-transfer question visible. Repeated bodies can have different supplied examples; they remain grouped and receive sensitivity analysis rather than silent reconciliation.

In [ ]:
audit = expanded["audit"]
counts = pd.DataFrame(audit["class_counts"])
counts["positive_rate"] = counts["sum"] / counts["size"]
counts["rule"] = counts.rule.str.split(":").str[0]
display(counts.rename(columns={"size": "Rows", "sum": "Violations"}).round(4))
display(pd.Series({name: audit[name] for name in ["development_rows", "repeated_body_policy_rows", "conflicting_groups", "conflicting_rows"]}, name="Development audit"))

## Two validation questions, one strict text boundary
**Familiar-policy CV** groups normalized comment bodies and stratifies by policy/target. **Held-out-policy CV** excludes each evaluated policy from training. Both remove training rows whose comment or any supplied example matches a validation comment. The assignments are frozen before model comparisons.

This purge substantially reduces available training data; the table makes that cost explicit. It cannot establish independence of paraphrases or common conversation origin, because conversation IDs, authors and timestamps are unavailable.

In [ ]:
display(pd.DataFrame(audit["folds"])[["protocol", "fold", "training_rows", "validation_rows", "purged_training_rows"]])

## Learned and target-derived features stay inside training
Vocabulary, IDF, scaling, reference percentiles, screening, NB weights and SVD use only each purged training partition. Target/context encodings use three inner grouped folds, inner support purging and inner-only priors. Unknown groups fall back to those training priors. Provided positive/negative examples are legitimate inference inputs, not the current row's target. Frozen encoders fit no competition labels.

The conflict sensitivity identifies conflicting groups from training labels only. The approximate-copy sensitivity uses character cosine ≥0.95, token Jaccard ≥0.90 and at least 40 characters, without target access. Both keep validation rows fixed; unchanged training sets reuse their primary fit.

In [ ]:
filters = pd.DataFrame(audit["training_sensitivities"])
filters["removed"] = filters.before - filters.after
display(filters.groupby(["protocol", "sensitivity", "model"])[["removed", "reused_primary"]].sum())

## What the metric and schema permit
The project reports **policy-macro ROC AUC** and pooled AUC separately. The official column-averaged AUC description and host per-policy attachment strongly corroborate equal rule weighting: 2,428 of 2,437 complete published rows agree within 1e-6. Nine discrepancies and six incomplete rows remain explicit; executable scorer parity and a project leaderboard score are not claimed.

Temporal, rolling, lag, season, team, opponent, coaching and external-rating variables are unavailable or inapplicable. Row order is not time. Current subreddit pages are not historical snapshots of supplied policies. External model revisions and licensing are documented; pretraining-overlap clearance is not claimed.

[02 · Feature research](02_baseline_and_review.ipynb) applies these boundaries to every compared representation.